In [3]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from collections import deque
import pygame
import time

def ajustar_deteccion_camara(model_path='modelfinal.h5'):
    # Inicializar pygame para sonido
    pygame.mixer.init()
    alarma_sonido = pygame.mixer.Sound("alerta.wav")  # Asegúrate de tener un archivo de sonido (wav)

    # Cargar el modelo sin compilar automáticamente
    model = load_model(model_path, compile=False)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    Q = deque(maxlen=15)  # Cola para promedio de predicciones
    UMBRAL_VIOLENCIA = 0.68  # Umbral para considerar violencia

    cap = cv2.VideoCapture(0)  # Abrir cámara

    if not cap.isOpened():
        print("Error al acceder a la cámara")
        return

    alarma_reproduciendose = False  # Bandera para verificar si la alarma ya está sonando
    tiempo_alarma_inicio = 0  # Tiempo de inicio de la alarma

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_display = frame.copy()  # Copia para mostrar

            # Preprocesamiento del frame para el modelo
            frame_proc = cv2.resize(frame, (128, 128))
            frame_proc = cv2.cvtColor(frame_proc, cv2.COLOR_BGR2RGB)
            frame_proc = frame_proc.astype("float32") / 255.0

            # Hacer predicción
            pred = model.predict(np.expand_dims(frame_proc, axis=0), verbose=0)[0][0]
            Q.append(pred)
            avg_pred = np.mean(Q)

            es_violencia = avg_pred > UMBRAL_VIOLENCIA

            # Color según resultado: rojo si violencia, verde si no
            if es_violencia:
                color = (0, 0, 255)  # Rojo
                texto = f"Violencia: True ({avg_pred:.2f})"

                if not alarma_reproduciendose:
                    alarma_sonido.play()  # Reproducir alarma
                    alarma_reproduciendose = True
                    tiempo_alarma_inicio = time.time()  # Marcar el tiempo de inicio de la alarma

            else:
                color = (0, 255, 0)  # Verde
                texto = f"Violencia: False ({avg_pred:.2f})"
                alarma_reproduciendose = False  # Detener la alarma si no es violencia

            # Detener la alarma después de 2 segundos
            if alarma_reproduciendose and (time.time() - tiempo_alarma_inicio) > 2:
                alarma_sonido.stop()  # Detener la alarma después de 2 segundos
                alarma_reproduciendose = False

            # Mostrar texto en pantalla
            cv2.putText(frame_display, texto, (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

            # Redimensionar para mejor visualización
            frame_display = cv2.resize(frame_display, (800, 800))
            cv2.imshow('Detección Ajustada', frame_display)

            # Salir con 'q' o 'ESC'
            if cv2.waitKey(1) in (ord('q'), 27):
                break

    finally:
        cap.release()
        cv2.destroyAllWindows()

ajustar_deteccion_camara()

C:\Users\Junior\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.3)
Hello from the pygame community. https://www.pygame.org/contribute.html
